# Sim-to-real on a Kaggle T4: domain randomization, a flywheel, and why the controller runs at the edge

Question: a controller trained in simulation fails on the real machine. How much do domain randomization and a sim-to-real flywheel close that gap, and why must the controller run at the edge?

Setup: two Tesla T4 GPUs (Turing, compute capability 7.5, driver 580, CUDA 13), internet on, NVIDIA Warp 1.19.0 on GPU 0 ([Warp docs](https://nvidia.github.io/warp/stable/)). The machine is a cart with a pole on top. The controller must keep the pole up for 6 seconds while random pushes hit it. The simulator is one Warp kernel. One GPU thread runs one whole episode, so thousands of machines run in parallel. A NumPy copy of the same step checks it. All randomness has a seed. Isaac Sim does not run here: it needs an RTX 4080 class GPU and driver 595.

The "real machine" is a second set of hidden physics inside the same simulator. It stands in for hardware. Only the functions `measure` and `collect_logs` run on it. No training step reads its numbers. The tables print the hidden truth only to show the fit error.

Exam objectives: physical.1 (physical AI vs digital AI, the three-computer pattern, edge inference) and physical.2 (simulation, digital twins, synthetic data, domain randomization, the sim-to-real gap, the sim-to-real flywheel).

Terms used below:

- **Physical AI**: AI that senses and acts in the physical world, such as a robot. **Digital AI** works only on data in software, such as a chat model.
- **Three-computer pattern**: NVIDIA splits a physical AI system over three computers. The training computer trains the models (for example NVIDIA DGX). The simulation computer simulates and tests them (for example NVIDIA OVX with Omniverse and Isaac Sim). The on-robot computer runs the model on the machine (for example NVIDIA Jetson).
- **Edge inference**: the trained model runs on the machine itself, not in a remote data center.
- **Simulation**: a program that computes how the machine moves. **Digital twin**: a simulation of one specific real machine, kept close to it with data. After the fit in part 5, the simulator is a digital twin of the hidden machine.
- **Synthetic data**: data that a simulator makes, not data measured on hardware. Every training episode here is synthetic.
- **Policy**: the function from sensor values to a motor command. Here it is a linear map of the 4 sensor values and the last 8 commands, followed by `tanh`.
- **Sim-to-real gap**: the loss of performance when a policy moves from simulation to the real machine. The cause is that the simulated physics differs from the real physics.
- **Domain randomization**: in training, you draw new physics values (mass, friction, motor strength, delay, noise) for every episode. The policy has to work on the whole range.
- **System identification**: you fit simulator values so that simulated motion matches logged real motion.
- **Sim-to-real flywheel**: a loop. Run on the real machine, log a few episodes, fit the simulator, narrow the randomization around the fit, retrain, run again.
- **Cross-entropy method (CEM)**: a search without gradients. Draw many candidates, score them, keep the best 15 percent, draw the next candidates near them.
- **Control period**: the time between two decisions. Here 20 ms. **Action delay**: the number of control steps between a command and its effect on the motor.
- **p50 and p99 latency**: half of the decisions are faster than p50. 99 percent are faster than p99.
- **World-step**: one machine advanced by one control step.

Constants I chose so that the task separates the methods. I set them in a NumPy prototype before I wrote this notebook. I looked only at task difficulty (push size, episode length, how far the real machine is from the nominal one). I did not tune any policy to the real machine.

- Pushes: each step has a 2 percent chance of a kick of up to 0.6 rad/s on the pole. Weaker pushes let the nominal policy succeed everywhere. Stronger pushes make every policy fail.
- Episode: 300 steps (6 s). Failure: pole angle above 0.4 rad or cart further than 2.4 m from the start.
- Hidden machine: pole 3 times heavier than nominal, friction 0.7, motor at half strength, 4 steps (80 ms) of delay, sensor noise 0.02. The policy sees past commands, so it can learn to cope with delay.
- The search budget is small on purpose (30 generations). A very large budget lets wide randomization cover more of the gap by itself on a task this small. Raise `CFG["gens"]` to test that.
- A caveat: the hidden machine follows the same equations as the simulator, with other numbers. The gap here is a gap in numbers only. A real robot also has effects that no simulator model contains.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Step | What you measure | Question to answer first |
|---|---|---|
| Simulator check | Warp vs NumPy difference after 50 steps; energy drift; replay error | Which differences are rounding, and which would mean a bug? |
| Speed | World-steps per second for 1 to 16,384 machines on CPU and GPU | At which machine count does the GPU start to win? Why is one machine slower on the GPU? |
| Training, 3 set-ups | Success in each set-up's own simulation | Does the nominal set-up look better or worse than the others in its own simulation? |
| Gap on the hidden machine | Success rate and survival time, own simulation vs real machine | Which set-up loses most? Is the narrow box around a wrong guess better than nominal? |
| Sweep of real machines | Success on a grid of pole mass and delay, nominal vs wide randomization | Where does each policy stop working? Does the wide box cost something on easy machines? |
| Flywheel, 3 turns | Real success and fit error per turn, and the number of real trajectories | Which physics values can 8 short logs identify? Does the control arm (more wide training, no real data) catch up? |
| Edge latency | p50 and p99 time per decision on one CPU thread | How many control periods does one decision cost? |
| Cloud delay | Real success with 0 to 200 ms extra action delay | Which extra delay breaks the policy? |

In [ ]:
# 1. Install NVIDIA Warp (pinned version) and show the GPUs
!pip install -q "warp-lang==1.19.0" 2>&1 | tail -2
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# 2. Constants and the NumPy reference simulator (pure NumPy, no GPU)
import json, os, subprocess, sys, time
import numpy as np, pandas as pd

QUICK = False        # True = tiny run to test the code on a CPU. Use False for the real run.
SEED = 42
f32 = np.float32
CFG = dict(pop=128, R=8, gens=30, seeds=5, n_eval=512, n_cell=64) if not QUICK else dict(pop=48, R=4, gens=12, seeds=2, n_eval=100, n_cell=24)
SID_POP, SID_GENS = (256, 20) if not QUICK else (64, 8)   # system identification: candidates, generations
T_EP = 300 if not QUICK else 150                          # steps per episode
T0, RESULTS = time.time(), []
def lap(name): print(f"[{(time.time() - T0) / 60:5.1f} min] {name}")
def add(section, setting, metric, value, note=""):
    RESULTS.append(dict(section=section, setting=setting, metric=metric, value=round(float(value), 4), note=note))

# Cart-pole constants. One control step = 20 ms = 2 physics sub-steps of 10 ms.
DT, SUB, G, FMAX = 0.02, 2, 9.81, 10.0
D, NH = 16, 8                  # size of the action ring buffer, and how many past actions the policy sees
TH_MAX, X_MAX = 0.4, 2.4       # an episode ends when the pole angle (rad) or the cart position (m) leaves this range
SC = np.array([1 / 2.4, 1 / 2.0, 1 / 0.4, 1 / 3.0], f32)   # scales the 4 sensor values to about +-1
NP = 4 + NH                    # policy weights: 4 sensor values + NH past actions, one linear map
PUSH_P, PUSH_S = 0.02, 0.6     # each step: chance 0.02 of a random kick of up to 0.6 rad/s on the pole

# Physics of one machine = one row of 7 numbers.
#                  cart_mass pole_mass half_length friction motor_gain delay_steps sensor_noise
NOMINAL  = np.array([1.0, 0.10, 0.5, 0.0, 1.0, 0.00, 0.000], f32)   # what the engineer believes
WIDE_LO  = np.array([0.5, 0.05, 0.3, 0.0, 0.3, 0.00, 0.00], f32)    # wide randomization: huge box
WIDE_HI  = np.array([2.0, 0.80, 0.9, 2.0, 1.5, 7.49, 0.08], f32)
WRONG_LO = np.array([0.9, 0.05, 0.5, 0.0, 1.0, 0.00, 0.000], f32)   # narrow box around a wrong guess:
WRONG_HI = np.array([1.1, 0.08, 0.5, 0.1, 1.2, 1.49, 0.005], f32)   # light pole, strong motor, no delay
REAL     = np.array([1.0, 0.30, 0.5, 0.7, 0.5, 4.00, 0.020], f32)   # the hidden machine: heavy pole, friction, weak motor, delay, noise

def physics_np(s, F, P):
    # One control step for n worlds. s: (n,4) = x, x_dot, theta, theta_dot. F: motor force (n,). P: (n,7).
    mc, mp, l, mu = P[:, 0], P[:, 1], P[:, 2], P[:, 3]
    h = f32(DT / SUB)
    x, xd, th, thd = s.T.copy()
    for _ in range(SUB):
        c, sn, M = np.cos(th), np.sin(th), mc + mp
        tmp = (F - mu * xd + mp * l * thd * thd * sn) / M
        ta = (G * sn - c * tmp) / (l * (f32(4 / 3) - mp * c * c / M))
        xa = tmp - mp * l * ta * c / M
        xd = xd + h * xa; thd = thd + h * ta; x = x + h * xd; th = th + h * thd   # semi-implicit Euler
    return np.stack([x, xd, th, thd], 1)

# mode 0 = train or test: the policy drives, the episode stops when the pole falls, the result is the survival time.
# mode 1 = log: the policy drives, no stop, and the first nlog sensor values and commands are recorded.
# mode 2 = replay: logged commands drive the machine from step t0. The result is the error against the logged cart motion.
def rollout_np(P, W, x0, seed, T, push_p, push_s, mode, act_in, obs_in, t0, nlog, expl):
    n = len(P); rng = np.random.default_rng(seed); ar = np.arange(n)
    d = (P[:, 5] + 0.5).astype(int); sg = P[:, 6:7]
    st = t0 if mode == 2 else np.zeros(n, int)
    s = (obs_in[ar, st] if mode == 2 else x0).astype(f32).copy()
    hist = np.zeros((n, D), f32); alive = np.ones(n, bool); steps = np.zeros(n, int); cost = np.zeros(n, f32)
    olog = np.zeros((n, max(nlog, 1), 4), f32); alog = np.zeros((n, max(nlog, 1)), f32)
    for k in range(T):
        if mode == 2:
            t = st + k
            if k >= 1:
                cost += (((s[:, :2] - obs_in[ar, t][:, :2]) * SC[:2]) ** 2).sum(1)
            idx = t - d
            u = np.where(idx >= 0, act_in[ar, np.maximum(idx, 0)], 0).astype(f32)
        else:
            obs = s + sg * rng.standard_normal((n, 4)).astype(f32)                    # noisy sensors
            z = np.concatenate([obs * SC, hist[:, [(k - 1 - j) % D for j in range(NH)]]], 1)
            a = np.tanh((z * W).sum(1)).astype(f32)                                    # the policy
            if expl > 0:
                a = np.clip(a + expl * rng.standard_normal(n).astype(f32), -1, 1)     # exploration noise
            hist[:, k % D] = a
            if k < nlog:
                olog[:, k] = obs; alog[:, k] = a
            u = hist[ar, (k - d) % D]                                                  # the command from d steps ago
            if push_p > 0:
                s[:, 3] += ((rng.random(n) < push_p) * (rng.random(n) * 2 - 1) * push_s).astype(f32)
        s = physics_np(s, FMAX * P[:, 4] * u, P)
        if mode == 0:
            alive &= (np.abs(s[:, 2]) < TH_MAX) & (np.abs(s[:, 0]) < X_MAX)
            steps += alive
            cost += np.where(alive, (s[:, 2] / TH_MAX) ** 2 + 0.5 * (s[:, 0] / X_MAX) ** 2, 0)
    return steps, cost, olog, alog

In [ ]:
# 3. The same step as a Warp kernel (one GPU thread = one whole episode), and one rollout() for all back-ends
try:
    import warp as wp
    wp.config.quiet = True
    HAS_GPU = wp.get_cuda_device_count() > 0
    print("warp", wp.__version__, "| CUDA devices:", wp.get_cuda_device_count())
except Exception as e:
    wp, HAS_GPU = None, False
    print("Warp is not available, using the NumPy back-end only:", repr(e)[:150])
BACKEND = "warp-gpu" if HAS_GPU else ("warp-cpu" if wp is not None else "numpy")
KERNEL_S = 0.0     # seconds spent in the last rollout() call (kernel only for Warp, everything for NumPy)

if wp is not None:
    C_DT, C_G, C_FMAX = wp.constant(DT), wp.constant(G), wp.constant(FMAX)
    C_SUB, C_D, C_NH = wp.constant(SUB), wp.constant(D), wp.constant(NH)
    C_THM, C_XM = wp.constant(TH_MAX), wp.constant(X_MAX)
    C_S0, C_S1, C_S2, C_S3 = (wp.constant(float(v)) for v in SC)

    @wp.func
    def cp_step(x: float, xd: float, th: float, thd: float, F: float, mc: float, mp: float, l: float, mu: float) -> wp.vec4:
        px = x
        pxd = xd
        pth = th
        pthd = thd
        h = C_DT / float(C_SUB)
        M = mc + mp
        for _s in range(C_SUB):
            c = wp.cos(pth)
            sn = wp.sin(pth)
            tmp = (F - mu * pxd + mp * l * pthd * pthd * sn) / M
            ta = (C_G * sn - c * tmp) / (l * (4.0 / 3.0 - mp * c * c / M))
            xa = tmp - mp * l * ta * c / M
            pxd = pxd + h * xa
            pthd = pthd + h * ta
            px = px + h * pxd
            pth = pth + h * pthd
        return wp.vec4(px, pxd, pth, pthd)

    @wp.kernel
    def rollout_k(P: wp.array2d[float], W: wp.array2d[float], x0: wp.array2d[float],
                  act_in: wp.array2d[float], obs_in: wp.array3d[float], t0: wp.array[int],
                  hist: wp.array2d[float], obs_out: wp.array3d[float], act_out: wp.array2d[float],
                  steps: wp.array[int], cost: wp.array[float],
                  seed: int, T: int, push_p: float, push_s: float, mode: int, nlog: int, expl: float):
        i = wp.tid()
        rng = wp.rand_init(seed, i)
        mc = P[i, 0]
        mp = P[i, 1]
        l = P[i, 2]
        mu = P[i, 3]
        gain = P[i, 4]
        sg = P[i, 6]
        d = int(P[i, 5] + 0.5)
        st = int(0)
        x = x0[i, 0]
        xd = x0[i, 1]
        th = x0[i, 2]
        thd = x0[i, 3]
        if mode == 2:
            st = t0[i]
            x = obs_in[i, st, 0]
            xd = obs_in[i, st, 1]
            th = obs_in[i, st, 2]
            thd = obs_in[i, st, 3]
        alive = int(1)
        n_ok = int(0)
        c_acc = float(0.0)
        for k in range(T):
            u = float(0.0)
            if mode == 2:
                t = st + k
                if k >= 1:
                    e0 = (x - obs_in[i, t, 0]) * C_S0
                    e1 = (xd - obs_in[i, t, 1]) * C_S1
                    c_acc += e0 * e0 + e1 * e1
                if t - d >= 0:
                    u = act_in[i, t - d]
            else:
                o0 = x + sg * wp.randn(rng)
                o1 = xd + sg * wp.randn(rng)
                o2 = th + sg * wp.randn(rng)
                o3 = thd + sg * wp.randn(rng)
                acc = W[i, 0] * o0 * C_S0 + W[i, 1] * o1 * C_S1 + W[i, 2] * o2 * C_S2 + W[i, 3] * o3 * C_S3
                for j in range(C_NH):
                    acc += W[i, 4 + j] * hist[i, (k - 1 - j + 512) % C_D]
                a = wp.tanh(acc)
                if expl > 0.0:
                    a = wp.clamp(a + expl * wp.randn(rng), -1.0, 1.0)
                hist[i, k % C_D] = a
                if k < nlog:
                    obs_out[i, k, 0] = o0
                    obs_out[i, k, 1] = o1
                    obs_out[i, k, 2] = o2
                    obs_out[i, k, 3] = o3
                    act_out[i, k] = a
                u = hist[i, (k - d + 512) % C_D]
                if push_p > 0.0:
                    if wp.randf(rng) < push_p:
                        thd = thd + (2.0 * wp.randf(rng) - 1.0) * push_s
            r = cp_step(x, xd, th, thd, C_FMAX * gain * u, mc, mp, l, mu)
            x = r[0]
            xd = r[1]
            th = r[2]
            thd = r[3]
            if mode == 0:
                if wp.abs(th) >= C_THM or wp.abs(x) >= C_XM:
                    alive = 0
                if alive == 1:
                    n_ok += 1
                    c_acc += (th / C_THM) * (th / C_THM) + 0.5 * (x / C_XM) * (x / C_XM)
                else:
                    break
        steps[i] = n_ok
        cost[i] = c_acc

def rollout_wp(P, W, x0, seed, T, push_p, push_s, mode, act_in, obs_in, t0, nlog, expl, device):
    global KERNEL_S
    n, L = len(P), max(nlog, 1)
    arr = lambda a: wp.array(np.ascontiguousarray(a, dtype=np.float32), dtype=float, device=device)
    ai = arr(act_in if act_in is not None else np.zeros((n, 1)))
    oi = arr(obs_in if obs_in is not None else np.zeros((n, 1, 4)))
    tt = wp.array((t0 if t0 is not None else np.zeros(n)).astype(np.int32), dtype=int, device=device)
    steps, cost = wp.zeros(n, dtype=int, device=device), wp.zeros(n, dtype=float, device=device)
    oo, ao = wp.zeros((n, L, 4), dtype=float, device=device), wp.zeros((n, L), dtype=float, device=device)
    hist = wp.zeros((n, D), dtype=float, device=device)
    args = [arr(P), arr(W), arr(x0), ai, oi, tt, hist, oo, ao, steps, cost,
            int(seed), int(T), float(push_p), float(push_s), int(mode), int(nlog), float(expl)]
    wp.synchronize_device(device)
    tic = time.perf_counter()
    wp.launch(rollout_k, dim=n, inputs=args, device=device)
    wp.synchronize_device(device)
    KERNEL_S = time.perf_counter() - tic
    return steps.numpy(), cost.numpy(), oo.numpy(), ao.numpy()

def rollout(P, W=None, x0=None, seed=0, T=None, push_p=None, push_s=None, mode=0, act_in=None, obs_in=None, t0=None, nlog=0, expl=0.0, backend=None):
    # n worlds. P: (n,7) physics per world. W: (n,NP) or (NP,) policy weights. Returns a dict (see the three modes above).
    global KERNEL_S
    backend = backend or BACKEND
    P = np.asarray(P, f32); n = len(P)
    W = np.zeros((n, NP), f32) if W is None else np.broadcast_to(np.asarray(W, f32), (n, NP))
    x0 = np.zeros((n, 4), f32) if x0 is None else np.asarray(x0, f32)
    T = T_EP if T is None else T
    push_p = PUSH_P if push_p is None else push_p
    push_s = PUSH_S if push_s is None else push_s
    seed = int(seed) % 2_147_483_647
    args = (P, W, x0, seed, T, push_p, push_s, mode, act_in, obs_in, t0, nlog, expl)
    if backend == "numpy":
        tic = time.perf_counter(); out = rollout_np(*args); KERNEL_S = time.perf_counter() - tic
    else:
        out = rollout_wp(*args, device="cuda:0" if backend == "warp-gpu" else "cpu")
    return dict(steps=out[0], cost=out[1], obs=out[2], act=out[3])

In [ ]:
# 4. Helpers: worlds, policy search (CEM), measurement, system identification, one flywheel run
def draw(lo, hi, n, rng):
    return (lo + (hi - lo) * rng.random((n, 7))).astype(f32)      # n machines, uniform in the box [lo, hi]
def x0s(n, rng):
    return rng.uniform(-0.1, 0.1, (n, 4)).astype(f32)             # small random start state

def train(lo, hi, seed, init=None):
    # Cross-entropy method. Each candidate policy runs on the same R random machines drawn from the box [lo, hi].
    # Fitness = survival share minus a small cost for leaning and drifting. Keep the best 15 percent, refit the Gaussian.
    pop, R, T = CFG["pop"], CFG["R"], T_EP
    rng = np.random.default_rng(seed)
    mean = np.zeros(NP, f32) if init is None else np.array(init, f32)
    sd = np.full(NP, 0.8 if init is None else 0.3, f32)
    curve = []
    for g in range(CFG["gens"]):
        C = (mean + sd * rng.standard_normal((pop, NP))).astype(f32)
        Pw, X = draw(lo, hi, R, rng), x0s(R, rng)
        r = rollout(np.tile(Pw, (pop, 1)), np.repeat(C, R, 0), np.tile(X, (pop, 1)), rng.integers(1 << 30), T)
        fit = (r["steps"] / T - 0.3 * r["cost"] / T).reshape(pop, R).mean(1)
        order = np.argsort(-fit)[:max(2, int(pop * 0.15))]
        mean, sd = C[order].mean(0), np.maximum(C[order].std(0), 0.05)
        curve.append(float(fit[order].mean()))
    return mean, curve

def measure(w, lo, hi, n, seed):
    # Success rate (pole up for the whole episode) and mean survival time in seconds, on n machines drawn from [lo, hi].
    rng = np.random.default_rng(seed)
    r = rollout(draw(lo, hi, n, rng), w, x0s(n, rng), rng.integers(1 << 30), T_EP)
    return float((r["steps"] == T_EP).mean()), float(r["steps"].mean() * DT)

# ---- system identification: fit mass, friction, motor gain and delay of the pole machine to logged real data
FREE = [1, 3, 4, 5]    # pole_mass, friction, motor_gain, delay. The cart mass and the pole length are known (a ruler and a scale).
N_LOG, W_LOG, EXPL, K_WIN, FRAC = 8, 30, 0.5, 8, [0.30, 0.20, 0.15]

def collect_logs(w, seed):
    # Run policy w (plus exploration noise) on the REAL machine. The rig does not stop when the pole falls.
    # Log the first W_LOG sensor values and commands of N_LOG episodes. This is the only real data the flywheel gets.
    rng = np.random.default_rng(seed)
    r = rollout(np.tile(REAL, (N_LOG, 1)), w, x0s(N_LOG, rng), rng.integers(1 << 30), W_LOG, mode=1, nlog=W_LOG, expl=EXPL)
    return r["obs"], r["act"]

def fit_physics(obs, act, seed):
    # Replay every K_WIN-step window of every log in many candidate machines. Score = error of cart position and speed.
    # Search with CEM. The pole angle is left out of the score: it is noisy and unstable, so it hides the other effects.
    rng = np.random.default_rng(seed); N, L = act.shape
    starts = np.arange(L - K_WIN + 1); J = N * len(starts)
    ob, ac, tt = np.repeat(obs, len(starts), 0), np.repeat(act, len(starts), 0), np.tile(starts, N)
    mean, sd = ((WIDE_LO + WIDE_HI) / 2)[FREE], ((WIDE_HI - WIDE_LO) / 3)[FREE]
    for g in range(SID_GENS):
        C = np.clip(mean + sd * rng.standard_normal((SID_POP, 4)), WIDE_LO[FREE], WIDE_HI[FREE]).astype(f32)
        P = np.tile(NOMINAL, (SID_POP, 1)); P[:, FREE] = C; P[:, 6] = 0
        r = rollout(np.repeat(P, J, 0), None, None, 0, K_WIN, push_p=0, mode=2,
                    act_in=np.tile(ac, (SID_POP, 1)), obs_in=np.tile(ob, (SID_POP, 1, 1)), t0=np.tile(tt, SID_POP))
        err = r["cost"].reshape(SID_POP, J).mean(1)
        best = C[np.argsort(err)[:max(4, SID_POP // 10)]]
        mean, sd = best.mean(0), np.maximum(best.std(0), (WIDE_HI - WIDE_LO)[FREE] * 0.01)
    return mean

def estimate_noise(obs):
    # The second difference of the cart position is mostly sensor noise: its spread is about sqrt(6) times the noise.
    d2 = obs[:, 2:, 0] - 2 * obs[:, 1:-1, 0] + obs[:, :-2, 0]
    return float(d2.std() / np.sqrt(6))

def narrow_box(est, noise, frac):
    # New randomization box: fitted value +- frac. Delay: fitted value +- 1 step. Noise: 0.5x to 1.5x the estimate.
    lo, hi = NOMINAL.copy(), NOMINAL.copy()
    for j, k in enumerate([1, 3, 4]):
        half = max(frac * abs(est[j]), 0.05 * (WIDE_HI - WIDE_LO)[k])
        lo[k], hi[k] = est[j] - half, est[j] + half
    lo[5], hi[5] = est[3] - 1.49, est[3] + 1.49
    lo[6], hi[6] = 0.5 * noise, 1.5 * noise
    return np.clip(lo, WIDE_LO, WIDE_HI), np.clip(hi, WIDE_LO, WIDE_HI)

def flywheel(seed, w0):
    # Three turns. Turn 0 is the wide-randomization policy w0. Control arm: more wide training, same compute, no real data.
    w, wc, LO, LA = w0, w0, [], []
    n_eval = CFG["n_eval"]
    ok0 = measure(w0, REAL, REAL, n_eval, 500 + seed)[0]
    rows = [dict(seed=seed, turn=0, n_traj=0, real_steps=0, real_ok=ok0, control_ok=ok0)]
    for k in range(3):
        o, a = collect_logs(w, 10_000 + 10 * seed + k)
        LO.append(o); LA.append(a)
        obs, act = np.concatenate(LO), np.concatenate(LA)
        est, noise = fit_physics(obs, act, 100 * seed + k), estimate_noise(obs)
        lo, hi = narrow_box(est, noise, FRAC[k])
        w, _ = train(lo, hi, 2000 + 10 * seed + k, init=w)
        wc, _ = train(WIDE_LO, WIDE_HI, 3000 + 10 * seed + k, init=wc)
        tru = REAL[FREE]                                  # the hidden truth is read here only to print the error
        rows.append(dict(seed=seed, turn=k + 1, n_traj=len(obs), real_steps=obs.shape[0] * obs.shape[1],
                         real_ok=measure(w, REAL, REAL, n_eval, 500 + seed + k + 1)[0],
                         control_ok=measure(wc, REAL, REAL, n_eval, 500 + seed + k + 1)[0],
                         err_mass=abs(est[0] - tru[0]) / tru[0], err_friction=abs(est[1] - tru[1]) / tru[1],
                         err_gain=abs(est[2] - tru[2]) / tru[2], err_delay_steps=abs(est[3] - tru[3]),
                         err_noise=abs(noise - REAL[6]) / REAL[6]))
    return rows, w

lap("helpers defined")

## Part 1: the simulator

State of one machine: cart position, cart speed, pole angle, pole speed. Seven numbers describe its physics: cart mass, pole mass, pole half-length, friction, motor gain, action delay (in steps), sensor noise. The motor force is `10 N x gain x command`, and the command comes from the policy `delay` steps ago. Each control step has 2 physics sub-steps with semi-implicit Euler. The policy sees noisy sensors and its own last 8 commands. A push changes the pole speed at random.

The next cells define:

1. Cell 2: the constants, the NumPy step, and the NumPy rollout loop.
2. Cell 3: the same step as a Warp kernel, and one `rollout()` function that runs on NumPy, Warp CPU or Warp GPU. It has three modes: train or test, log, and replay.
3. Cell 4: the helpers for policy search, measurement, system identification and the flywheel.

Then cell 5 checks the simulator. It asserts four things: Warp and NumPy agree within 2e-3 on 6 machines with different delays; energy stays constant without friction and input; cart momentum stays constant; a replay with the true delay reproduces the log, and a replay with one more step of delay does not.

In [ ]:
# 5. Check the simulator: Warp vs NumPy, energy, delay buffer, and replay
print("back-end:", BACKEND)
rng = np.random.default_rng(SEED); n = 6
P = draw(WIDE_LO, WIDE_HI, n, rng); P[:, 5] = [0, 1, 2, 3, 4, 5]; P[:, 6] = 0       # delays 0..5, no sensor noise
W = rng.normal(0, 0.5, (n, NP)).astype(f32); X = x0s(n, rng)
args = dict(W=W, x0=X, seed=1, T=50, push_p=0, push_s=0, mode=1, nlog=50)
ref = rollout(P, backend="numpy", **args)
rollout(P, **args)                                   # first call compiles the Warp kernel
new = rollout(P, **args)
dif = max(np.abs(ref["obs"] - new["obs"]).max(), np.abs(ref["act"] - new["act"]).max())
print(f"max |Warp - NumPy| over 6 worlds x 50 steps x (4 sensors + 1 command): {dif:.2e}")
assert dif < 2e-3, "Warp and NumPy disagree"
add("simulator", "Warp vs NumPy, 50 steps", "max_abs_difference", dif, BACKEND)

# Energy and momentum: no friction, no motor, no delay. The pole hangs down and swings from 0.5 rad off the bottom
# (an upright pole falls, and the fall would amplify the integrator error). Energy must stay nearly constant.
p = np.array([[1.0, 0.1, 0.5, 0.0, 1.0, 0, 0]], f32); s = np.array([[0, 0, np.pi - 0.5, 0]], f32)
def energy(s, p):
    mc, mp, l = p[0, :3]; x, xd, th, thd = s[0]
    return 0.5 * (mc + mp) * xd ** 2 + mp * l * xd * thd * np.cos(th) + (2 / 3) * mp * l ** 2 * thd ** 2 + mp * G * l * np.cos(th)
def momentum(s, p):
    mc, mp, l = p[0, :3]; x, xd, th, thd = s[0]
    return (mc + mp) * xd + mp * l * thd * np.cos(th)
e0, drift, mom = energy(s, p), 0.0, 0.0
for _ in range(100):
    s = physics_np(s, np.zeros(1, f32), p); drift = max(drift, abs(energy(s, p) - e0)); mom = max(mom, abs(momentum(s, p)))
swing = 0.1 * G * 0.5 * (1 + np.cos(np.pi - 0.5))    # energy that moves between height and motion
print(f"energy drift over 2 s: {drift / swing:.2%} of the swing energy | max cart momentum: {mom:.1e}")
assert drift / swing < 0.05 and mom < 1e-2
add("simulator", "no friction, no input", "energy_drift_share", drift / swing)

# Delay buffer and replay: log a run with random commands. Replay it with the true delay: the error is about 0.
# Replay it with one step more or less delay: the error grows. So the ring buffer and the replay agree.
log = rollout(P, W=W, x0=X, seed=2, T=40, push_p=0, mode=1, nlog=40, expl=0.5)
def replay_err(Pq):
    t0 = np.full(n, 10)
    return rollout(Pq, push_p=0, T=K_WIN, mode=2, act_in=log["act"], obs_in=log["obs"], t0=t0)["cost"]
Pm = P.copy(); Pm[:, 5] += 1
e_true, e_off = replay_err(P), replay_err(Pm)
print("replay error, true delay :", np.round(e_true, 7))
print("replay error, delay + 1  :", np.round(e_off, 7))
assert e_true.max() < 1e-4 < np.median(e_off), "delay buffer and replay disagree"
add("simulator", "replay with true delay", "max_error", e_true.max())
lap("simulator checked")

## Part 2: simulation speed

The kernel runs a whole episode per thread, so the GPU has no Python loop between steps. The next cell times 300 steps for 1, 64, 1024, 4096 and 16,384 machines on three back-ends: NumPy on the CPU, the same Warp kernel on the CPU, and the same Warp kernel on the GPU. It reports the best of 3 runs. For Warp the time covers the kernel only, not the copy of arrays to and from the device. Compile time is outside the timing (a warm-up call runs first).

In [ ]:
# 6. Simulation speed: world-steps per second. One world-step = one machine advanced by one 20 ms control step.
T_SP = 300
backends = ["numpy"] + (["warp-cpu"] if wp is not None else []) + (["warp-gpu"] if HAS_GPU else [])
counts = [1, 64, 1024, 4096, 16384] if not QUICK else [1, 64, 1024]
rng = np.random.default_rng(SEED); rows = []
for b in backends:
    rollout(draw(WIDE_LO, WIDE_HI, 8, rng), rng.normal(0, 0.5, (8, NP)), x0s(8, rng), 0, 10, mode=1, backend=b)   # warm-up, compiles
    for n in counts:
        P, W, X = draw(WIDE_LO, WIDE_HI, n, rng), rng.normal(0, 0.5, (n, NP)).astype(f32), x0s(n, rng)
        best = min((rollout(P, W, X, 7, T_SP, mode=1, backend=b), KERNEL_S)[1] for _ in range(3))
        rows.append(dict(backend=b, worlds=n, seconds=best, world_steps_per_s=n * T_SP / best))
sp = pd.DataFrame(rows)
base = sp[sp.backend == "numpy"].set_index("worlds").world_steps_per_s
sp["x_vs_numpy"] = sp.world_steps_per_s / sp.worlds.map(base)
with pd.option_context("display.float_format", "{:,.4g}".format, "display.width", 200):
    print(sp.to_string(index=False))
for r in sp.itertuples():
    add("speed", f"{r.backend} | {r.worlds} worlds", "world_steps_per_s", r.world_steps_per_s, f"{r.x_vs_numpy:.1f}x NumPy")
lap("speed done")

## Part 3: train three set-ups with the same budget

All three use CEM with the same budget: generations x policies x machines per policy x steps. Each policy runs on the same set of machines in a generation, so the comparison is fair. Only the box of physics values differs.

- **(a) Nominal only**: every machine is the nominal one (light pole, no friction, no delay, no noise).
- **(b) Wide domain randomization**: every machine is drawn from a large box. The hidden machine is inside the box, but the policy does not know where.
- **(c) Narrow box around a wrong guess**: a light pole, a strong motor and almost no delay. The hidden machine is outside this box.

Fitness is the share of the episode survived, minus a small cost for leaning and drifting. The cost lets CEM rank policies that all survive.

In [ ]:
# 7. Train three set-ups with the same budget (population x worlds x generations) and several seeds
SETUPS = {"a_nominal": (NOMINAL, NOMINAL), "b_wide_dr": (WIDE_LO, WIDE_HI), "c_narrow_wrong": (WRONG_LO, WRONG_HI)}
POL, rows = {k: [] for k in SETUPS}, []
for name, (lo, hi) in SETUPS.items():
    for s in range(CFG["seeds"]):
        w, curve = train(lo, hi, SEED + s)
        POL[name].append(w)
        own = measure(w, lo, hi, CFG["n_eval"], 100 + s)         # sim: the worlds this set-up trained on
        nom = measure(w, NOMINAL, NOMINAL, CFG["n_eval"], 200 + s)
        real = measure(w, REAL, REAL, CFG["n_eval"], 300 + s)
        rows.append(dict(setup=name, seed=s, fit_first_gen=curve[0], fit_last_gen=curve[-1],
                         own_sim_ok=own[0], own_sim_s=own[1], nominal_ok=nom[0], real_ok=real[0], real_s=real[1]))
    lap(f"trained {name}")
TR = pd.DataFrame(rows)
print(f"budget per set-up: {CFG['gens']} generations x {CFG['pop']} policies x {CFG['R']} machines x {T_EP} steps, {CFG['seeds']} seeds")
print(TR.groupby("setup")[["fit_first_gen", "fit_last_gen"]].mean().round(3))

## Part 4: the sim-to-real gap

Each trained policy now runs in two places: in its own simulation, and on the hidden machine. The gap is the difference. The second table sweeps real machines over pole mass and delay. The hidden machine is the cell with pole 0.3 kg and delay 4. In the other cells, friction, motor strength and noise stay as on the hidden machine.

In [ ]:
# 8. The sim-to-real gap: success rate and survival time in simulation and on the real machine (mean and std over seeds)
g = TR.groupby("setup")
gap = pd.DataFrame({
    "own_sim_success": g.own_sim_ok.mean(), "nominal_sim_success": g.nominal_ok.mean(), "real_success": g.real_ok.mean(),
    "real_success_std": g.real_ok.std(), "own_sim_survival_s": g.own_sim_s.mean(), "real_survival_s": g.real_s.mean()})
gap["gap_success (own sim - real)"] = gap.own_sim_success - gap.real_success
gap["gap_survival_s"] = gap.own_sim_survival_s - gap.real_survival_s
print(f"episode length {T_EP * DT:.0f} s. Success = pole up for the whole episode.")
print(gap.round(3).to_string())
for name, r in gap.iterrows():
    add("sim-to-real gap", name, "own_sim_success", r.own_sim_success); add("sim-to-real gap", name, "real_success", r.real_success)
    add("sim-to-real gap", name, "real_survival_s", r.real_survival_s); add("sim-to-real gap", name, "gap_success", r["gap_success (own sim - real)"])

# Sweep of real machines: pole mass x delay. The other real physics (friction, motor, noise) stay as on the hidden machine.
masses, delays = [0.1, 0.2, 0.3, 0.4, 0.5], [0, 1, 2, 3, 4, 5, 6]
def sweep(name):
    out = pd.DataFrame(index=[f"pole {m} kg" for m in masses], columns=[f"delay {d}" for d in delays], dtype=float)
    for m in masses:
        for d in delays:
            p = REAL.copy(); p[1], p[5] = m, d
            out.loc[f"pole {m} kg", f"delay {d}"] = np.mean([measure(w, p, p, CFG["n_cell"], 900 + i)[0] for i, w in enumerate(POL[name])])
    return out
HEAT = {k: sweep(k) for k in ("a_nominal", "b_wide_dr")}
for k, h in HEAT.items():
    print(f"\nSuccess rate on real machines, policy set-up {k} (mean over seeds). The hidden machine is pole 0.3 kg, delay 4.")
    try:
        display(h.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format("{:.2f}"))
    except Exception:
        print(h.round(2).to_string())
    add("sweep", k, "mean_success_over_grid", h.values.mean())
lap("gap measured")

## Part 5: the sim-to-real flywheel

One turn of the loop:

1. Run the current policy on the real machine, with some exploration noise. Log 8 episodes of 30 steps. This is the only real data. Real data is expensive, so the cap grows by 8 episodes per turn (8, 16, 24 in total).
2. Fit the simulator to all logs so far (system identification). The fit has four unknowns: pole mass, friction, motor gain and delay. The cart mass and the pole length are known. The fit uses the same parallel simulator in replay mode. Every 8-step window of every log is replayed in many candidate machines. The score is the error of cart position and cart speed. Sensor noise comes from the second difference of the logged cart position.
3. Narrow the randomization box around the fit: plus or minus 30, 20 and 15 percent in turns 1, 2 and 3, and plus or minus 1 step of delay.
4. Retrain from the current policy with the same budget as one training run.
5. Measure on the real machine. These test episodes are not used for training.

Control arm: the same extra training with the wide box and no real data. It shows whether the gain comes from real data or only from more compute.

Turn 0 is the wide-randomization policy (b) of the same seed.

In [ ]:
# 9. Sim-to-real flywheel: 3 turns. Real data per turn: N_LOG logged episodes of W_LOG steps. Nothing else from the real machine trains anything.
fw_rows, FINAL_W = [], []
for s in range(CFG["seeds"]):
    rows_s, w_final = flywheel(s, POL["b_wide_dr"][s])
    fw_rows += rows_s; FINAL_W.append(w_final)
    lap(f"flywheel seed {s}")
FW = pd.DataFrame(fw_rows)
tab = FW.groupby("turn").agg(real_trajectories=("n_traj", "mean"), real_steps_logged=("real_steps", "mean"),
                             real_success=("real_ok", "mean"), real_success_std=("real_ok", "std"),
                             control_success=("control_ok", "mean"), control_success_std=("control_ok", "std"),
                             err_mass=("err_mass", "mean"), err_friction=("err_friction", "mean"), err_gain=("err_gain", "mean"),
                             err_delay_steps=("err_delay_steps", "mean"), err_noise=("err_noise", "mean"))
print("Turn 0 = the wide-randomization policy. control = the same extra training with wide randomization and no real data.")
print("err_* = relative error of the fitted value against the hidden truth (delay: error in steps).")
print(tab.round(3).to_string())
print("\nHidden truth (printed for the reader; no step above used it): pole mass %.2f, friction %.2f, motor gain %.2f, delay %d steps, noise %.3f" % tuple(REAL[[1, 3, 4, 5, 6]]))
for t, r in tab.iterrows():
    add("flywheel", f"turn {t}", "real_success", r.real_success, f"{int(r.real_trajectories)} real trajectories")
    add("flywheel", f"turn {t}", "control_success_no_real_data", r.control_success)
    if t > 0:
        add("flywheel", f"turn {t}", "err_mass", r.err_mass); add("flywheel", f"turn {t}", "err_friction", r.err_friction)
        add("flywheel", f"turn {t}", "err_gain", r.err_gain); add("flywheel", f"turn {t}", "err_delay_steps", r.err_delay_steps)
lap("flywheel done")

## Part 6: edge inference

Two questions. First: how long does one decision take on one CPU thread, compared with the 20 ms control period? Second: what happens if the decision runs in the cloud? A round trip adds action delay. The simulator shows the effect: it adds 0, 20, 50, 100 and 200 ms (0, 1, 3, 5 and 10 control steps) on top of the 4 steps of the real machine. 50 ms is 2.5 steps and rounds up to 3.

The next markdown cell maps the three computers to the cells of this notebook.

In [ ]:
# 10. Edge inference: one decision of the trained policy on ONE CPU thread. Compare with the 20 ms control period.
try:
    os.sched_setaffinity(0, {0})
    print("pinned to CPU 0")
except Exception as e:
    print("could not pin to one CPU:", repr(e)[:80])
w = FINAL_W[0].astype(f32)
N_DEC = 20000 if not QUICK else 2000
stream = np.random.default_rng(SEED).normal(0, 0.1, (N_DEC + 500, 4)).astype(f32)
past, lat = np.zeros(NH, f32), np.empty(N_DEC)
for i in range(N_DEC + 500):
    tic = time.perf_counter_ns()
    z = np.concatenate([stream[i] * SC, past])          # sensors + past commands
    a = np.tanh(w @ z)                                  # the decision
    past = np.concatenate([[a], past[:-1]]).astype(f32)
    if i >= 500:
        lat[i - 500] = time.perf_counter_ns() - tic    # the first 500 decisions are warm-up
p50, p99, mx = (np.percentile(lat, q) / 1e6 for q in (50, 99, 100))
print(f"decision latency, {N_DEC} decisions: p50 {p50:.4f} ms | p99 {p99:.4f} ms | max {mx:.3f} ms | control period {DT * 1000:.0f} ms")
print(f"p99 uses {p99 / (DT * 1000):.3%} of the control period (compute only: a real robot adds sensor and motor I/O)")
add("edge", "one CPU thread, NumPy", "latency_p50_ms", p50); add("edge", "one CPU thread, NumPy", "latency_p99_ms", p99)
add("edge", "one CPU thread, NumPy", "latency_max_ms", mx)

# A cloud round trip adds delay to every command. Add it on top of the 4 steps of the real machine.
rows = []
for ms in [0, 20, 50, 100, 200]:
    extra = int(ms / (DT * 1000) + 0.5)
    p = REAL.copy(); p[5] += extra
    assert p[5] < D, "ring buffer too small"
    ok = [measure(wf, p, p, CFG["n_eval"], 700 + i) for i, wf in enumerate(FINAL_W)]
    rows.append(dict(extra_round_trip_ms=ms, extra_steps=extra, total_delay_steps=int(p[5]),
                     success=np.mean([o[0] for o in ok]), success_std=np.std([o[0] for o in ok]), survival_s=np.mean([o[1] for o in ok])))
    add("edge", f"+{ms} ms action delay", "real_success", rows[-1]["success"])
print("\nFinal flywheel policy on the real machine with extra action delay (mean over seeds):")
print(pd.DataFrame(rows).round(3).to_string(index=False))
lap("edge done")

### The three computers in this notebook

| Computer | Job in physical AI | Where it is in this notebook | In the NVIDIA stack |
|---|---|---|---|
| Training computer | Trains and retrains the policy | `train()` (CEM) in cells 4 and 7, and the retraining in the flywheel (cell 9) | For example NVIDIA DGX |
| Simulation computer | Runs many machines, makes synthetic data, tests policies, fits the digital twin | The Warp kernel (cell 3), the speed test (cell 6), the replay fit `fit_physics()` (cells 4 and 9) | For example NVIDIA OVX with Omniverse and Isaac Sim (Isaac Sim does not run on this T4) |
| On-robot computer | Runs the trained policy at the control rate, next to the sensors and motors | The one-thread NumPy decision loop (cell 10) | For example NVIDIA Jetson |

On Kaggle all three jobs share one T4 and one VM. In a real system they are three machines. Only the on-robot computer sits inside the control loop.

In [ ]:
# 11. Summary table, also written to /kaggle/working/sim_to_real_results.csv
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
df = pd.DataFrame(RESULTS)
df.to_csv(f"{OUT}/sim_to_real_results.csv", index=False)
with pd.option_context("display.max_rows", 500, "display.width", 200, "display.max_colwidth", 60):
    print(df.to_string(index=False))
print(f"\ntotal wall time {(time.time() - T0) / 60:.1f} min | saved {OUT}/sim_to_real_results.csv")

## What to look for

- **Simulator check.** The Warp and NumPy difference should sit near float32 rounding. The energy drift shows the integrator error. The replay test shows that the delay buffer and the replay use the same delay.
- **Speed.** Compare the GPU with the CPU at each machine count. One machine cannot fill a GPU. A launch has a fixed cost. The speed-up appears when there are enough machines to hide it.
- **Training.** Compare the first and last generation fitness. Then compare each set-up in its own simulation. A policy can look good in its own simulation and still fail on the real machine.
- **Gap.** Compare the own-simulation column with the real column for each set-up. Compare (a) with (c): is a narrow box around a wrong guess closer to the real machine than the nominal one? Then compare (b) in its own simulation with (b) on the real machine. The wide box contains machines that may be impossible to control. What does that do to its own-simulation number? Read the std over seeds: a difference smaller than the std is noise.
- **Sweep.** Find the edge of each policy: the pole mass and delay at which success falls. Check the easy corner (light pole, small delay) for a cost of the wide box.
- **Flywheel.** Compare the flywheel with the control arm at each turn. If both rise, part of the gain is only more training. Read the fit errors: friction and motor gain may be harder to identify than delay. The number of real trajectories is small and capped on purpose.
- **Edge.** Compare p99 with the 20 ms period. Then read the cloud table. A delay of several steps changes the physics the policy must handle. A policy trained at 4 steps of delay did not see 14.
- This run uses 5 seeds. The hidden machine has the same equations as the simulator. A real robot has more unmodeled effects, so the gap on real hardware can be larger than the gap here.

## Exam rules this proves

1. **physical.1**: Physical AI senses and acts in the world and must run inside a control loop with a fixed period. The trained policy runs on the on-robot computer (edge inference). A cloud round trip adds action delay, and the measured success rate falls as the delay grows.
2. **physical.1**: The three-computer pattern is a training computer, a simulation computer and an on-robot computer. In this notebook they map to the CEM training cells, the Warp simulator and replay cells, and the one-thread decision loop.
3. **physical.2**: A simulator runs many machines in parallel on a GPU and makes synthetic data. A policy trained only on nominal physics scores well in its own simulation and loses success on a machine with different physics. That loss is the sim-to-real gap.
4. **physical.2**: Domain randomization trains on a wide range of physics so that the real machine falls inside the range. A narrow range around a wrong guess keeps the real machine outside the range, so the policy does not transfer.
5. **physical.2**: The sim-to-real flywheel uses a small number of real logs to fit the simulator (a digital twin), narrows the randomization around the fit, and retrains. Compare it with a control arm that trains the same amount without real data.

## Optional: Newton

Newton is an open physics engine built on Warp ([docs](https://newton-physics.github.io/newton/)). The next cell installs it into a separate folder, runs its built-in `basic_pendulum` example without a viewer, in a subprocess with a timeout, and prints the result. If anything fails, the cell prints a note and the notebook is still complete. Nothing above depends on this cell.

In [ ]:
# 12. OPTIONAL: run a built-in Newton example headless in a subprocess. Any failure prints a note and the notebook goes on.
NEWTON_DIR = f"{OUT}/newton_pkgs"
try:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", NEWTON_DIR, "newton[examples]", "warp-lang==1.19.0"],
                       capture_output=True, text=True, timeout=900)
    print("pip install exit code:", r.returncode, r.stderr[-300:] if r.returncode else "")
    if r.returncode == 0:
        env = dict(os.environ, PYTHONPATH=NEWTON_DIR + os.pathsep + os.environ.get("PYTHONPATH", ""))
        tic = time.time()
        r = subprocess.run([sys.executable, "-m", "newton.examples", "basic_pendulum", "--viewer", "null", "--num-frames", "100"],
                           capture_output=True, text=True, timeout=300, env=env)
        print(f"newton basic_pendulum: exit code {r.returncode} in {time.time() - tic:.0f} s")
        print((r.stdout + r.stderr)[-600:])
except Exception as e:
    print("Newton cell skipped:", repr(e)[:200])